# 02 · Preprocesamiento
## Segmentación Inteligente de Pacientes — Programa de Medicina Funcional · Comfama

| | |
|---|---|
| **Propósito** | Convertir las respuestas crudas del notebook 01 en las 19 features numéricas/binarias que consume el modelo, e imputar valores faltantes. |
| **Entrada** | `data/_processed/01_raw_clean.csv` |
| **Salida** | `data/_processed/02_features.csv` (features sin escalar + medianas de imputación). |

Estas mismas 19 features (incluyendo las claves y la forma de derivarlas) están
centralizadas en `app/domain/ml_features.py` para que el modelo entrenado use
**exactamente la misma lógica** al servir predicciones en producción sobre las
respuestas del cuestionario de la app (evita *train/serve skew*).

In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path("..").resolve()))
from app.domain.ml_features import FEATURE_NAMES, CARDIO_FEATURES, DIGEST_FEATURES

PROCESSED_DIR = Path("../data/_processed")
df = pd.read_csv(PROCESSED_DIR / "01_raw_clean.csv")
print(f"Filas: {len(df):,}")


Filas: 16,174


### Ingeniería de features
- **Hábitos saludables / síntomas digestivos**: se cuentan las opciones marcadas (excluyendo "Ninguno").
- **Actividad física**: binaria (la realiza o no), tipo de fuerza (sí/no) y frecuencia semanal a un número representativo.
- **Estrés, ansiedad, nerviosismo, técnica de manejo**: Sí/No → 1/0.
- **Medicamentos**: se buscan subcadenas (`hipoglicemiante`, `hipolipemiante`, `antiácido`/`bomba de protones`) porque el campo es texto libre separado por `;`.
- **Mediciones y laboratorios**: se dejan como vienen (numéricos).

In [2]:
def has_any(text, needles):
    t = str(text).lower()
    return any(n in t for n in needles)

df["n_habitos_saludables"] = df["habitos_alim"].apply(
    lambda t: 0 if "ninguno" in str(t).lower() else len([p for p in str(t).split(";") if p.strip()])
)
df["hace_actividad"] = (~df["actividad_tipo"].str.lower().str.contains("ninguno", na=False)).astype(int)
df["actividad_fuerza"] = df["actividad_tipo"].apply(lambda t: int(has_any(t, ["fuerza"])))

freq_map = {"ninguno": 0, "1 vez": 1, "2 veces": 2, "3 veces": 3, "4 veces": 4, "5 o más veces": 5}
df["frecuencia_actividad_num"] = (
    df["actividad_frecuencia"].str.strip().str.lower().str.rstrip(";").map(freq_map).fillna(0)
)

df["estres_alto_bin"] = (df["estres_alto"].str.strip().str.lower() == "sí").astype(int)
df["ansioso_bin"] = (df["ansioso"].str.strip().str.lower() == "sí").astype(int)
df["nervioso_bin"] = (df["nervioso"].str.strip().str.lower() == "sí").astype(int)
df["tecnica_estres_bin"] = (df["tecnica_estres"].str.strip().str.lower() == "sí").astype(int)

df["usa_hipoglicemiante"] = df["medicamentos"].apply(lambda t: int(has_any(t, ["hipoglicemiante"])))
df["usa_hipolipemiante"] = df["medicamentos"].apply(lambda t: int(has_any(t, ["hipolipemiante"])))
df["usa_antiacido_ibp"] = df["medicamentos"].apply(
    lambda t: int(has_any(t, ["antiácid", "antiacid", "bomba de protones"]))
)
df["n_sintomas_gi"] = df["sintomas_gi"].apply(
    lambda t: 0 if "ninguno" in str(t).lower() else len([p for p in str(t).split(";") if p.strip()])
)

print("Features derivadas:", FEATURE_NAMES)
df[FEATURE_NAMES].describe().round(2).T


Features derivadas: ['imc', 'perimetro_abdominal', 'pa_sistolica', 'pa_diastolica', 'colesterol_total', 'colesterol_hdl', 'hba1c', 'usa_hipoglicemiante', 'usa_hipolipemiante', 'n_sintomas_gi', 'usa_antiacido_ibp', 'estres_alto_bin', 'ansioso_bin', 'nervioso_bin', 'tecnica_estres_bin', 'hace_actividad', 'actividad_fuerza', 'frecuencia_actividad_num', 'n_habitos_saludables']


,count,mean,std,min,25%,50%,75%,max
imc,16174.0,28.39,5.70,14.44,24.34,27.48,31.85,65.57
perimetro_abdominal,16174.0,90.81,13.73,41.00,81.00,90.00,99.00,175.00
pa_sistolica,16174.0,118.34,7.87,84.00,118.00,118.00,120.00,190.00
pa_diastolica,16174.0,77.01,5.62,50.00,76.00,78.00,80.00,120.00
colesterol_total,16174.0,193.05,40.51,67.00,167.00,192.00,215.00,423.00
colesterol_hdl,16174.0,49.25,11.76,19.00,41.00,48.00,55.00,133.00
hba1c,16174.0,5.59,0.59,4.00,5.30,5.50,5.70,14.40
usa_hipoglicemiante,16174.0,0.14,0.35,0.00,0.00,0.00,0.00,1.00
usa_hipolipemiante,16174.0,0.26,0.44,0.00,0.00,0.00,1.00,1.00
n_sintomas_gi,16174.0,0.63,1.45,0.00,0.00,0.00,0.00,11.00


### Imputación
Las 7 mediciones/laboratorios continuos (`imc`, `perimetro_abdominal`,
`pa_sistolica`, `pa_diastolica`, `colesterol_total`, `colesterol_hdl`,
`hba1c`) están completas en el histórico (0 nulos), así que en este dataset
la imputación no actúa. Se calcula y guarda igual la mediana de cada
feature: en producción, una consulta que aún no tiene mediciones del médico
(p. ej. justo después del cuestionario del paciente) sí llega con nulos, y
`TrainedClassifier` los reemplaza por estas medianas (o usa un puntaje
neutral si falta *toda* la información de un eje clínico — ver notebook 04).

In [3]:
medians = df[FEATURE_NAMES].median(numeric_only=True)
print(medians.round(2))

nulls_before = df[FEATURE_NAMES].isna().sum().sum()
print()
print(f"Nulos en el histórico (deberían ser 0): {nulls_before}")


imc                          27.48
perimetro_abdominal          90.00
pa_sistolica                118.00
pa_diastolica                78.00
colesterol_total            192.00
colesterol_hdl               48.00
hba1c                         5.50
usa_hipoglicemiante           0.00
usa_hipolipemiante            0.00
n_sintomas_gi                 0.00
usa_antiacido_ibp             0.00
estres_alto_bin               0.00
ansioso_bin                   0.00
nervioso_bin                  0.00
tecnica_estres_bin            0.00
hace_actividad                1.00
actividad_fuerza              0.00
frecuencia_actividad_num      0.00
n_habitos_saludables          2.00
dtype: float64

Nulos en el histórico (deberían ser 0): 0


### Vista previa del feature set final

In [4]:
out_path = PROCESSED_DIR / "02_features.csv"
df[["documento"] + FEATURE_NAMES].to_csv(out_path, index=False)
print(f"Guardado: {out_path} ({len(df):,} filas, {len(FEATURE_NAMES)} features)")
df[FEATURE_NAMES].head(5)


Guardado: ..\data\_processed\02_features.csv (16,174 filas, 19 features)


,imc,perimetro_abdominal,pa_sistolica,pa_diastolica,colesterol_total,colesterol_hdl,hba1c,usa_hipoglicemiante,usa_hipolipemiante,n_sintomas_gi,usa_antiacido_ibp,estres_alto_bin,ansioso_bin,nervioso_bin,tecnica_estres_bin,hace_actividad,actividad_fuerza,frecuencia_actividad_num,n_habitos_saludables
0,31.588613,113.0,118,78.0,152,47,5.9,0,0,0,0,0,1,1,0,1,0,0,0
1,29.270994,100.0,118,78.0,199,55,5.6,0,1,0,0,0,0,0,0,0,0,0,2
2,34.209026,99.0,118,78.0,200,52,5.9,0,0,0,0,0,1,0,0,1,0,0,0
3,19.477147,62.0,118,78.0,180,61,5.2,0,0,0,0,0,1,1,0,1,0,0,0
4,22.675737,63.0,118,78.0,160,48,5.2,0,0,0,0,0,1,0,0,1,0,0,2


---
**Resumen del notebook 02:** 19 features derivadas (9 del eje cardiometabólico,
2 del eje digestivo, 8 de hábitos/estilo de vida), sin nulos que imputar en
el histórico. La misma función de derivación se reutiliza en producción vía
`app/domain/ml_features.py`.
➡️ Continúa en **`03_entrenamiento_clustering`** para el EDA de segmentación
y la búsqueda de *k*.